In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain_teddynote import logging

from dotenv import load_dotenv

load_dotenv(override=True)

# 추적을 위한 프로젝트 이름 설정
logging.langsmith("Samsung-Asset-AI-Portal")

# 프롬프트

### 시스템 프롬프트

In [ ]:
_SYSTEM_PROMPT = """당신은 자산운용사에서 변액일임펀드 설정/해지 지시서 처리를 담당하는 오퍼레이터 입니다.
    당신의 역할은 수익자가 메일로 보내온 변액일임펀드 설정/해지 지시서를 시스템에 입력하기 전에
    변액일임펀드 설정/해지 지시서에서 확정분과 청구분을 구분하여 확정분에 대한 설정/해지 데이터와 청구분에 대한 설정/해지 데이터를 수집하고 정리하는 역할입니다.
"""

_SYSTEM_PROMPT_ENG = """You are an operator at an asset management company responsible for processing variable managed fund setup/termination instructions.

Your role is to, before entering the instructions into the system, 
distinguish between confirmed items and pending/claim items in the variable managed fund setup/termination instruction sent by the beneficiary via email, 
and collect and organize the setup/termination data for both the confirmed portion and the pending/claim portion.
"""


### PDF text to markdown 프롬프트

In [ ]:
#pdf 텍스트를 markdown 형식으로 변환하는 프롬프트
_CREATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT = """
  - docling 추출 data를 기반으로 전체 내용을 분석하세요.

  - 모든 필드와 모든 데이터 및 모든 텍스트를 최대한 빠짐없이 모두 정리하세요.(설정/해지와 상관없는 텍스트도 모두 정리할 것)

  - 한자가 발견되면 반드시 한글로 변환하세요.

  - 통합 또는 요약하지 말고, 종목(펀드)명과 펀드코드 단위로 data를 정리하세요.

  - 모든 메타 데이터, 테이블 컬럼, 필드들의 의미와 기능을 분석하여 정규화 하고 테이블로 정리하세요.

  - 모든 텍스트(종목(펀드)명과 펀드코드와 상관없는 텍스트 포함)들도 의미를 분석하여 정규화 하고 테이블로 정리하세요.

  - 각 단어와 코드, 숫자 데이터들을 pdfplumber 추출 data와 비교하여 보다 정확한 data를 선택하세요.

  - PDF에서 추출한 data의 특성상 인접한 컬럼의 데이터가 중복되거나, 인접한 컬럼으로 병합되는 오류가 발생할 수 있습니다. 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.

  - PDF에서 추출한 data의 특성상 마지막 row의 데이터가 상위 row의 데이터와 병합되는 오류가 발생할 수 있습니다. 테이블의 흐름을 분석하여 정리 결과의 테이블에서 마지막 row의 데이터가 상위 row의 데이터와 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요. 병합되어 있으면 수정하세요.

  - PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

  - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.

  - 종목명에서 약어를 사용하는지 판단하여 약어를 유지하세요.

  - 펀드 코드와 펀드명이 정확한지 확인하고 작성하세요.(합계 행과 혼동되지 않도록 주의)

  - 펀드 코드를 기준으로 펀드 개수를 집계하세요.(펀드 코드가 없는 경우 펀드 개수를 집계하지 않음)

  - 설정건이 존재할 경우, 데이터의 맥락과 의미를 분석하여 설정건에 대한 매입통보일자를 반드시 추출하세요.

  - 해지건이 존재할 경우, 데이터의 맥락과 의미를 분석하여 해지건에 대한 환매신청일을 반드시 추출하세요.

  - 문서 제목을 발견하면 최상단에 출력하세요.

  - 원문을 번역하지 말고 원문 그대로 출력하세요.
"""


_VALIDATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT = """
  - 한자가 발견되면 반드시 한글로 변환하세요.

  - 모든 필드와 텍스트가 정확히 추출되었는지 확안하세요.

  - 정리 결과에서 누락된 필드와 데이터가 있는지 확인하세요.

  - 펀드코드를 기준으로 펀드 개수와 거래 건수가 정확하게 집계되었는지 확안하세요.

  - 펀드코드를 기준으로 펀드 행과 합계 행이 정확히 구분되어 집계되었는지 확안하세요.

  - 모든 항목에서 종목명(펀드명)과 펀드코드가 정확하게 작성되었는지 확안하세요.

  - 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.

  - 설정건이 존재할 경우, 설정건에 대한 매입통보일자가 추출되었는지 반드시 확안하세요.

  - 해지건이 존재할 경우, 해지건에 대한 환매신청일자가 추출되었는지 반드시 확안하세요.

  - PDF에서 추출한 data의 특성상 인접한 컬럼의 데이터가 중복되거나, 인접한 컬럼으로 병합되는 오류가 발생할 수 있습니다. 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요.

  - PDF에서 추출한 data의 특성상 마지막 row의 데이터가 상위 row의 데이터와 병합되는 오류가 발생할 수 있습니다. 테이블의 흐름을 분석하여 정리 결과의 테이블에서 마지막 row의 데이터가 상위 row의 데이터와 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요.

  - PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - Markdown 코드의 오류 여부를 검수하여 오류가 발견되면 수정하세요.

  - 검수 결과에서 오류가 발견되면 오류 항목을 수정하세요.
"""


# pdf 텍스트를 markdown 형식으로 변환하는 프롬프트
def get_prompt_pdf_text_to_markdown(pdf_text_docling: str, pdf_text_pdfplumber: str) -> str:
  prompt_text = f"""
  아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 PDF 파일에서 pdfplumber와 docling 라이브러리를 사용하여 추출한 data입니다.


  아래의 추출 data를 LLM 모델이 잘 이해할 수 있도록 정리 지침에 따라 정리하세요.

  정리 지침에 따라 정리한 내용을 검수 지침에 따라 검수하세요.

  검수 지침에 따라 검수한 내용을 markdown 형식으로 작성하세요.



  # 변액일임펀드 설정/해지 지시서 PDF 파일 내용 - docling 라이브러리 사용

  {pdf_text_docling}



  # 변액일임펀드 설정/해지 지시서 PDF 파일 내용 - pdfplumber 라이브러리 사용

  {pdf_text_pdfplumber}



  # 반드시 지켜야 할 정리 지침

  {_CREATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT}


  # 검수 지침
  {_VALIDATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT}

  """
  return prompt_text

  

### excel, text to markdown

In [ ]:
_CREATE_TEXT_TO_MARKDOWN_RULE_PROMPT = """
  - 추출 data를 기반으로 전체 내용을 분석하세요.

  - 모든 필드와 모든 데이터 및 모든 텍스트를 최대한 빠짐없이 모두 정리하세요.(설정/해지와 상관없는 텍스트도 모두 정리할 것)

  - 한자가 발견되면 반드시 한글로 변환하세요.

  - 통합 또는 요약하지 말고, 종목(펀드)명과 펀드코드 단위로 data를 정리하세요.

  - 모든 메타데이터, 테이블 컬럼, 필드들의 의미와 기능을 분석하여 정규화 하고 테이블로 정리하세요.

  - 모든 텍스트(종목(펀드)명과 펀드코드와 상관없는 텍스트 포함)들도 의미를 분석하여 정규화 하고 테이블로 정리하세요.

  - 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 확인하세요. 중복 또는 병합되어 있으면 수정하세요.

  - 테이블의 흐름을 분석하여 정리 결과의 테이블에서 마지막 row의 데이터가 상위 row의 데이터와 병합되어 작성되어 있는지 확인하세요. 병합되어 있으면 수정하세요.

  - 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

  - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.

  - 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.

  - 종목명에서 약어를 사용하는지 판단하여 약어를 유지하세요.

  - 펀드 코드와 펀드명이 정확한지 확인하고 작성하세요.(합계 행과 혼동되지 않도록 주의)

  - 펀드 코드를 기준으로 펀드 개수를 집계하세요.(펀드 코드가 없는 경우 펀드 개수를 집계하지 않음)

  - 설정건이 존재할 경우, 데이터의 맥락과 의미를 분석하여 설정건에 대한 매입통보일자를 반드시 추출하세요.

  - 해지건이 존재할 경우, 데이터의 맥락과 의미를 분석하여 해지건에 대한 환매신청일을 반드시 추출하세요.

  - 문서 제목을 발견하면 최상단에 출력하세요.

  - 원문을 번역하지 말고 원문 그대로 출력하세요.
"""

_VALIDATE_TEXT_TO_MARKDOWN_RULE_PROMPT = """
  - 한자가 발견되면 반드시 한글로 변환하세요.

  - 모든 필드와 텍스트가 정확히 추출되었는지 확인한다.

  - 정리 결과에서 누락된 필드와 데이터가 있는지 확인하세요.

  - 펀드코드를 기준으로 펀드 개수와 거래 건수가 정확하게 집계되었는지 확인한다.

  - 펀드코드를 기준으로 펀드 행과 합계 행이 정확히 구분되어 집계되었는지 확인한다.

  - 모든 항목에서 종목명(펀드명)과 펀드코드가 정확하게 작성되었는지 확인한다.

  - 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
  
  - 설정건이 존재할 경우, 설정건에 대한 매입통보일자가 추출되었는지 반드시 확안한다.

  - 해지건이 존재할 경우, 해지건에 대한 환매신청일자가 추출되었는지 반드시 확안한다.

  - 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 확인하세요.

  - 테이블의 흐름을 분석하여 정리 결과의 테이블에서 마지막 row의 데이터가 상위 row의 데이터와 병합되어 작성되어 있는지 확인하세요.

  - 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하세요.

  - Markdown 코드의 오류 여부를 검수하여 오류가 발견되면 수정한다.

  - 검수 결과에서 오류가 발견되면 오류 항목을 수정하세요.
"""


def get_prompt_text_to_markdown(original_text: str):
  prompt_text = f"""
  아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 파일에서 추출한 data입니다.

  아래의 추출 data를 LLM 모델이 잘 이해할 수 있도록 정리 지침에 따라 정리하세요.

  정리 지침에 따라 정리한 내용을 검수 지침에 따라 검수하세요.

  검수 지침에 따라 검수한 내용을 markdown 형식으로 작성하세요.



  ### 변액일임펀드 설정/해지 지시서 파일 내용 ###

  {original_text}



  # 반드시 지켜야 할 중요 지침

  {_CREATE_TEXT_TO_MARKDOWN_RULE_PROMPT}


  # 검수 지침
  {_VALIDATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT}

  """
  return prompt_text

### PDF to markdown validation 프롬프트

In [ ]:
# pdf 텍스트를 markdown 형식으로 변환한 데이터가 지침에 따라 올바르게 작성되었는지 검수하는 프롬프트
def get_prompt_pdf_text_to_markdown_validate(text_to_markdown: str, pdf_text_pdfplumber: str, pdf_text_docling: str) -> str:
    prompt_text = f"""
    아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 PDF 파일에서 text를 추출하여 markdown 형식으로 정리하여 작성한 data입니다.

    원본 PDF 파일에서 추출한 text와 비교하여 markdown 형식으로 정리한 data가 아래의 작성 지침에 따라 올바르게 작성되었는지 검수 지침에 따라 검수하세요.

    검수가 완료되면 검수 결과가 반영된 markdown을 반드시 출력형식에 따라 출력하세요.


    # 변액일임펀드 설정/해지 지시서 markdown 형식 정리 내용

    {text_to_markdown}


    # 변액일임펀드 설정/해지 지시서 PDF 파일 내용 - docling 라이브러리 사용

    {pdf_text_docling}
   

    # 변액일임펀드 설정/해지 지시서 PDF 파일 내용 - pdfplumber 라이브러리 사용

    {pdf_text_pdfplumber}


    # 검수 지침(반드시 수행)

    {_VALIDATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT} 


    # 작성 지침

    {_CREATE_PDF_TEXT_TO_MARKDOWN_RULE_PROMPT}


    # 출력 형식 (***반드시 순서대로 markdown 형식으로 출력하고 아래의 내용만 출력할 것***)

      1. 매입통보일 / 환매신청일

      2. 거래 지시 내용 테이블

      3. 전체 거래 집계 현황 테이블

      4. 전체 펀드코드 개수, 거래 개수 집계 현황 테이블

      5. 메타데이터 설명 테이블

    """
    return prompt_text

### excel, text to markdown validation 프롬프트

In [ ]:

def get_prompt_text_to_markdown_validate(text_to_markdown: str, original_text: str):
  prompt_text = f"""
    아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 파일에서 text를 추출하여 markdown 형식으로 정리하여 작성한 data입니다.

    원본 파일에서 추출한 text와 비교하여 markdown 형식으로 정리한 data가 아래의 작성 지침에 따라 올바르게 작성되었는지 검수 지침에 따라 검수하세요.

    검수가 완료되면 검수 결과가 반영된 markdown을 출력하세요.


    # 변액일임펀드 설정/해지 지시서 markdown 형식 정리 내용

    {text_to_markdown}


    # 변액일임펀드 설정/해지 지시서 파일 내용

    {original_text}


    # 검수 지침(반드시 수행)

    {_VALIDATE_TEXT_TO_MARKDOWN_RULE_PROMPT} 


    # 작성 지침

    {_CREATE_TEXT_TO_MARKDOWN_RULE_PROMPT}


    # 출력 형식 (***반드시 순서대로 markdown 형식으로 출력하고 아래의 내용만 출력할 것***)

      1. 매입통보일 / 환매신청일

      2. 거래 지시 내용 테이블

      3. 전체 거래 집계 현황 테이블

      4. 전체 펀드코드 개수, 거래 개수 집계 현황 테이블

      5. 메타데이터 설명 테이블
      
  """
  return prompt_text

# LLM 모델 생성

In [ ]:
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")

def create_llm_model():

    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:",
        temperature=0.0,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.1로 설정
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY
    )
    # llm = init_chat_model(
    #     "openai:gpt-4o",
    #     temperature=0.0,
    #     top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
    # )
    return llm

# LLM - pdf text to markdown 변환

In [ ]:
def convert_pdf_text_to_markdown(document_text_docling: str, document_text_pdfplumber: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_pdf_text_to_markdown(document_text_docling, document_text_pdfplumber)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response